In [1]:
!pip -q install opencv-python-headless scikit-image timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.7 MB/s eta 0:00:00


In [2]:
%%writefile p3_base.py
import os, math, glob, random
from dataclasses import dataclass
from typing import Tuple, List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50

# ---------- Data ----------
def make_loaders(data_root: str, batch_size: int = 16, num_segments: int = 50):
    """
    Expects:
      data_root/
        train/ real/ fake/
        valid/ real/ fake/
        test/  real/ fake/
    """
    size = 320
    train_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    train_ds = ImageFolder(os.path.join(data_root, "train"), transform=train_tf)
    valid_ds = ImageFolder(os.path.join(data_root, "valid"), transform=eval_tf)
    test_ds  = ImageFolder(os.path.join(data_root, "test"),  transform=eval_tf)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_dl, valid_dl, test_dl

# ---------- Small helpers ----------
class GlobalAvgPool(nn.Module):
    def forward(self, x):  # (B,C,H,W) -> (B,C)
        return x.mean(dim=(2,3))

# ---------- FFT adapter (stage-1: simple + light) ----------
class FFTAdapter(nn.Module):
    """
    Takes a feature map (B,C,H,W). Computes 2D FFT per channel.
    Produces magnitude & phase embeddings, mixes with 1x1 conv,
    and returns same-channel feature map so we can fuse later.
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.reduce_mag = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.reduce_phase = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.mix = nn.Conv2d(out_ch*2, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        # FFT
        # torch.fft.fft2 -> complex, keep same spatial size
        X = torch.fft.fft2(x, dim=(-2,-1))
        mag = torch.abs(X)          # magnitude
        phase = torch.angle(X)      # phase in radians
        
        # normalize (log for magnitude stabilizes)
        mag = torch.log1p(mag)
        # standardize per-batch
        for t in (mag, phase):
            mean = t.mean(dim=(2,3), keepdim=True)
            std = t.std(dim=(2,3), keepdim=True) + 1e-6
            t.sub_(mean).div_(std)

        m = self.reduce_mag(mag)
        p = self.reduce_phase(phase)
        f = torch.cat([m, p], dim=1)
        f = self.mix(f)
        f = self.bn(f)
        f = self.act(f)
        return f

# ---------- P3-ish backbone + tiny encoder ----------
class TinyTransformerEncoder(nn.Module):
    """
    Very small encoder that operates on pooled tokens from ResNet feature map.
    In stage-1 we will usually FREEZE this, so it just passes information forward.
    """
    def __init__(self, dim=1024, depth=2, heads=8, mlp_ratio=2.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=int(dim*mlp_ratio),
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)

    def forward(self, x):  # x: (B,T,dim)
        return self.enc(x)

class P3Model(nn.Module):
    def __init__(self, use_fft=False, fft_fuse='concat',
                 freeze_backbone=False, freeze_encoder=False, num_segments=50):
        super().__init__()
        self.use_fft = use_fft
        self.fft_fuse = fft_fuse

        # ResNet backbone (ImageNet weights can be loaded from the runner)
        self.backbone = resnet50(weights=None)
        self.backbone.fc = nn.Identity()  # keep feature map path
        # last conv output channels = 2048, feature map ~ (B,2048,10,10) for 320x320

        # a small 1x1 conv to bring channels down for the encoder
        self.neck = nn.Conv2d(2048, 1024, kernel_size=1, bias=False)
        self.neck_bn = nn.BatchNorm2d(1024)
        self.neck_act = nn.ReLU(inplace=True)

        # tokenization: simple grid pooling -> one token per spatial cell
        self.pool_tokens = nn.AdaptiveAvgPool2d((4,4))  # 16 tokens
        self.encoder = TinyTransformerEncoder(dim=1024, depth=2, heads=8, mlp_ratio=2.0)

        # FFT branch that taps AFTER neck (same spatial size)
        if use_fft:
            self.fft = FFTAdapter(in_ch=1024, out_ch=1024)

        # classifier head
        if use_fft and fft_fuse == 'concat':
            head_dim = 1024*2
        else:
            head_dim = 1024
        self.gap = GlobalAvgPool()
        self.head = nn.Sequential(
            nn.Linear(head_dim, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 2)
        )

        # freezing knobs
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False
            for p in self.neck.parameters(): p.requires_grad = False
            for p in self.neck_bn.parameters(): p.requires_grad = False
        if freeze_encoder:
            for p in self.encoder.parameters(): p.requires_grad = False

    def forward(self, x):
        # ResNet stem + blocks -> (B,2048,H/32,W/32)
        feats = []
        # run backbone up to layer4 while preserving feature map
        x = self.backbone.conv1(x); x = self.backbone.bn1(x); x = self.backbone.relu(x); x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x); x = self.backbone.layer2(x); x = self.backbone.layer3(x); x = self.backbone.layer4(x)

        h = self.neck_act(self.neck_bn(self.neck(x)))  # (B,1024,h,w)

        # tiny transformer over 4x4 tokens
        tokens = self.pool_tokens(h)                   # (B,1024,4,4)
        B,C,H,W = tokens.shape
        tokens = tokens.reshape(B, C, H*W).permute(0,2,1)  # (B,16,1024)
        z = self.encoder(tokens)                           # (B,16,1024)
        # put tokens back to a map and interpolate to h's size
        z_map = z.permute(0,2,1).reshape(B, C, H, W)
        z_map = F.interpolate(z_map, size=h.shape[-2:], mode='bilinear', align_corners=False)

        # FFT branch (frozen backbone; fft is the new learnable part)
        if self.use_fft:
            f = self.fft(h.detach() if (not any(p.requires_grad for p in self.neck.parameters())) else h)
            if self.fft_fuse == 'add':
                fused = z_map + f
            else:  # concat
                fused = torch.cat([z_map, f], dim=1)
        else:
            fused = z_map

        # global pooling + head
        g = self.gap(fused)
        logits = self.head(g)
        return logits

# ---------- Training ----------
@torch.no_grad()
def accuracy(logits, y):
    pred = logits.argmax(1)
    return (pred == y).float().mean().item()

def train_one_epoch(model, dl, optimizer, device):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    ce = nn.CrossEntropyLoss()
    for x,y in dl:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = ce(logits, y)
        loss.backward()
        optimizer.step()
        b = x.size(0)
        total_loss += loss.item()*b
        total_acc  += accuracy(logits.detach(), y)*b
        n += b
    return total_loss/n, total_acc/n

@torch.no_grad()
def evaluate(model, dl, device):
    model.eval()
    ce = nn.CrossEntropyLoss()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x,y in dl:
        x,y = x.to(device), y.to(device)
        logits = model(x)
        loss = ce(logits, y)
        b = x.size(0)
        total_loss += loss.item()*b
        total_acc  += accuracy(logits, y)*b
        n += b
    return total_loss/n, total_acc/n

# ---------- I/O ----------
def save_ckpt(model, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model": model.state_dict()}, path)

def load_ckpt(model, path, strict=True):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"], strict=strict)

def build_p3_model(use_fft=False, fft_fuse='concat', freeze_backbone=False, freeze_encoder=False, num_segments=50):
    return P3Model(use_fft=use_fft, fft_fuse=fft_fuse,
                   freeze_backbone=freeze_backbone, freeze_encoder=freeze_encoder,
                   num_segments=num_segments)


Writing p3_base.py


In [3]:
%%writefile run_p3.py
import argparse, os, torch
import torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights

from p3_base import (
    make_loaders, build_p3_model, train_one_epoch, evaluate, save_ckpt
)

def load_ckpt_shape_compatible(model, path):
    ckpt = torch.load(path, map_location="cpu")["model"]
    msd = model.state_dict()
    keep = {k:v for k,v in ckpt.items() if k in msd and v.shape == msd[k].shape}
    missing = [k for k in msd.keys() if k not in keep]
    skipped = [k for k in ckpt.keys() if k not in keep]
    model.load_state_dict(keep, strict=False)
    print(f"Loaded {len(keep)} tensors; skipped {len(skipped)} (shape mismatch).")

def main():
    p = argparse.ArgumentParser(description="P-3 baseline with optional FFT")
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--batch_size', type=int, default=16)
    p.add_argument('--epochs', type=int, default=5)
    p.add_argument('--lr', type=float, default=1e-3)
    p.add_argument('--use_fft', action='store_true')
    p.add_argument('--fft_fuse', type=str, default='concat', choices=['concat','add'])
    p.add_argument('--freeze_backbone', action='store_true')
    p.add_argument('--freeze_encoder', action='store_true')
    p.add_argument('--ckpt', type=str, default='')
    p.add_argument('--save', type=str, default='/kaggle/working/p3_fft.pth')
    p.add_argument('--imagenet_backbone', action='store_true')
    args = p.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    train_dl, valid_dl, test_dl = make_loaders(args.data_root, batch_size=args.batch_size)

    model = build_p3_model(use_fft=args.use_fft, fft_fuse=args.fft_fuse,
                           freeze_backbone=args.freeze_backbone, freeze_encoder=args.freeze_encoder).to(device)

    if args.imagenet_backbone:
        try:
            print("Loading ImageNet weights into ResNet…")
            model.backbone.load_state_dict(resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).state_dict(), strict=False)
            print("✓ Loaded.")
        except Exception as e:
            print(f"Could not load ImageNet weights: {e}")

    if args.ckpt and os.path.isfile(args.ckpt):
        print(f"Loading compatible weights from {args.ckpt}")
        load_ckpt_shape_compatible(model, args.ckpt)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=args.lr, weight_decay=1e-4)

    best = 0.0
    for ep in range(1, args.epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_dl, opt, device)
        va_loss, va_acc = evaluate(model, valid_dl, device)
        print(f"Epoch {ep:02d} | train {tr_loss:.4f}/{tr_acc:.4f} | valid {va_loss:.4f}/{va_acc:.4f}")
        if va_acc > best:
            best = va_acc
            save_ckpt(model, args.save)
            print(f"  ↳ Saved best to {args.save}")

    te_loss, te_acc = evaluate(model, test_dl, device)
    print(f"Test: loss {te_loss:.4f} acc {te_acc:.4f}")

if __name__ == "__main__":
    main()


Writing run_p3.py


In [4]:
DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
!ls -R $DATA_ROOT | head -n 40


/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake:
test
train
valid

/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test:
fake
real

/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test/fake:
00276TOPP4.jpg
008BYSE725.jpg
009ZTJ3621.jpg
00F8LKY6JC.jpg
00JEP4Z36Z.jpg
00KEKJJ1Q4.jpg
00MZYXAT77.jpg
00PB1BNIE8.jpg
00QKZTHTLF.jpg
00V5CZZSSO.jpg
00XUQJZGHU.jpg
01050DBM3C.jpg
010MNNNOZS.jpg
017EMQKL4D.jpg
01877XHF3A.jpg
01EA4QAE1L.jpg
01FCG0FF23.jpg
01IUWPPCNT.jpg
01KDGEL6IQ.jpg
01MI46B2OH.jpg
01RUCFVMJY.jpg
025AJ3S2VH.jpg
025TMVT114.jpg
028M3M2AAR.jpg
02DWMIB1T5.jpg
02NUKFGPSJ.jpg
02P1HEQ0GB.jpg
02TPLQKRQB.jpg
02XAKN4F4U.jpg
02YD6CVUGS.jpg
ls: write error: Broken pipe


In [5]:
!python run_p3.py \
  --data_root $DATA_ROOT \
  --batch_size 32 --epochs 1 --lr 1e-3 \
  --freeze_backbone --freeze_encoder \
  --save /kaggle/working/p3_base.pth


Epoch 01 | train 0.6748/0.5808 | valid 0.6683/0.5947
  ↳ Saved best to /kaggle/working/p3_base.pth
Test: loss 0.6693 acc 0.5880


In [6]:
!python run_p3.py \
  --data_root $DATA_ROOT \
  --batch_size 32 --epochs 2 --lr 5e-4 \
  --use_fft --fft_fuse concat \
  --freeze_backbone --freeze_encoder \
  --ckpt /kaggle/working/p3_base.pth \
  --save /kaggle/working/p3_fft.pth


Loading compatible weights from /kaggle/working/p3_base.pth
Loaded 351 tensors; skipped 1 (shape mismatch).
Epoch 01 | train 0.6627/0.6027 | valid 0.6549/0.6133
  ↳ Saved best to /kaggle/working/p3_fft.pth
Epoch 02 | train 0.6474/0.6236 | valid 0.6435/0.6277
  ↳ Saved best to /kaggle/working/p3_fft.pth
Test: loss 0.6448 acc 0.6287


In [7]:
!python run_p3.py --data_root $DATA_ROOT --epochs 0 \
  --ckpt /kaggle/working/p3_base.pth


Loading compatible weights from /kaggle/working/p3_base.pth
Loaded 352 tensors; skipped 0 (shape mismatch).
Test: loss 0.6693 acc 0.5880


In [8]:
!python run_p3.py --data_root $DATA_ROOT --epochs 0 \
  --use_fft --fft_fuse concat \
  --ckpt /kaggle/working/p3_fft.pth


Loading compatible weights from /kaggle/working/p3_fft.pth
Loaded 360 tensors; skipped 0 (shape mismatch).
Test: loss 0.6448 acc 0.6287


In [9]:
import torch, numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from p3_base import build_p3_model, load_ckpt

tf = transforms.Compose([
    transforms.Resize((320,320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
test_ds = datasets.ImageFolder(f"{DATA_ROOT}/test", transform=tf)
test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def eval_ckpt(path, use_fft):
    m = build_p3_model(use_fft=use_fft, fft_fuse='concat', freeze_backbone=True, freeze_encoder=True).to(device)
    load_ckpt(m, path, strict=False)
    m.eval()
    ys, ps, yh = [], [], []
    with torch.no_grad():
        for x,y in test_dl:
            x = x.to(device)
            logits = m(x)
            prob_fake = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            yh.append((prob_fake >= 0.5).astype(int))
            ps.append(prob_fake); ys.append(y.numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); yh = np.concatenate(yh)
    acc = accuracy_score(ys, yh)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, yh, average='binary', zero_division=0)
    auc = roc_auc_score(ys, ps)
    cm = confusion_matrix(ys, yh)
    return acc, prec, rec, f1, auc, cm

print("Baseline:")
acc, prec, rec, f1, auc, cm = eval_ckpt("/kaggle/working/p3_base.pth", use_fft=False)
print(f"acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}  auc={auc:.4f}\ncm=\n{cm}\n")

print("FFT:")
acc, prec, rec, f1, auc, cm = eval_ckpt("/kaggle/working/p3_fft.pth", use_fft=True)
print(f"acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}  f1={f1:.4f}  auc={auc:.4f}\ncm=\n{cm}")


Baseline:
acc=0.5880  prec=0.6271  rec=0.4341  f1=0.5131  auc=0.6356
cm=
[[7419 2581]
 [5659 4341]]

FFT:
acc=0.6287  prec=0.6064  rec=0.7337  f1=0.6640  auc=0.6850
cm=
[[5237 4763]
 [2663 7337]]
